# Convolutional networks

`nn.Conv2d` slides learned filters over `N × C × H × W` images.

- `in_channels` / `out_channels`: depth in / depth out
- `kernel_size`, `stride`, `padding`

After convolutions we **flatten** spatial dimensions before `nn.Linear`.


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)


## 1. One conv layer, then a linear head


In [ ]:
conv_layer = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
input_data = torch.randn(64, 3, 32, 32)  # N, C, H, W
conv_output = conv_layer(input_data)
print("conv output:", conv_output.shape)

flattened = conv_output.view(conv_output.size(0), -1)
print("flattened:", flattened.shape)  # 64 × 16384

linear = nn.Linear(in_features=flattened.size(1), out_features=10)
print("logits:", linear(flattened).shape)


## 2. Peek at feature maps


In [ ]:
indices = torch.randint(0, 64, (6,))
images = conv_output[indices]

fig, axes = plt.subplots(1, 6, figsize=(12, 4))
for i, ax in enumerate(axes.flat):
    channel_ids = torch.randint(0, 16, (3,))
    rgb = images[i][channel_ids]
    rgb = (rgb - rgb.amin(dim=(1, 2), keepdim=True)) / (
        rgb.amax(dim=(1, 2), keepdim=True) - rgb.amin(dim=(1, 2), keepdim=True) + 1e-8
    )
    ax.imshow(rgb.permute(1, 2, 0).detach().cpu().numpy())
    ax.set_title(f"img {indices[i].item()}\nch {channel_ids.tolist()}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## 3. Shapes through conv → ReLU → pool → flatten

Print shapes inside `forward` until they become second nature. This is the fastest way to debug CNNs.


In [ ]:
class ShapeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)  # 28x28 -> 24x24
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)  # 24x24 -> 12x12
        self.fc1 = nn.Linear(10 * 12 * 12, 50)

    def forward(self, x):
        print("input:", tuple(x.shape))
        x = self.pool(self.relu(self.conv1(x)))
        print("after conv/pool:", tuple(x.shape))
        x = x.view(x.size(0), -1)
        print("after flatten:", tuple(x.shape))
        x = self.fc1(x)
        print("after fc:", tuple(x.shape))
        return x


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ShapeNet().to(device)
dummy = torch.randn(1, 1, 28, 28, device=device)
_ = model(dummy)


## 4. A compact CNN we can train


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(8 * 14 * 14, num_classes)

    def forward(self, x):
        x = self.pool(self.relu(self.conv(x)))
        x = x.view(x.size(0), -1)
        return self.fc(x)


cnn = SimpleCNN()
logits = cnn(torch.randn(4, 1, 28, 28))
print(logits.shape)


## 5. Train the compact CNN and log to TensorBoard

Synthetic 28×28 images and random labels — the accuracy will not be high, but the loop and logging are the real lesson.

After running the cell:

```bash
tensorboard --logdir runs
```


In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter

X = torch.randn(512, 1, 28, 28)
y = torch.randint(0, 10, (512,))
loader = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)

cnn = SimpleCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn.parameters(), lr=1e-3)
writer = SummaryWriter("runs/cnn_experiment")

global_step = 0
for epoch in range(3):
    correct = total = 0
    for inputs, labels in loader:
        optimizer.zero_grad()
        outputs = cnn(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        writer.add_scalar("Loss/train", loss.item(), global_step)
        global_step += 1
        predicted = outputs.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    acc = correct / total
    writer.add_scalar("Accuracy/train", acc, epoch)
    print(f"Epoch {epoch + 1}  last loss {loss.item():.4f}  acc {acc:.3f}")

writer.close()
print("Logs written to runs/cnn_experiment")
